In [6]:
import finnhub
import os
import json
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

data = finnhub_client.company_earnings(symbol="AAPL")

In [7]:
# How many quarters returned
print(f"Quarters returned: {len(data)}")

# Inspect each quarter
for quarter in data:
    print(quarter)

Quarters returned: 4
{'actual': 2.84, 'estimate': 2.7257, 'period': '2025-12-31', 'quarter': 1, 'surprise': 0.1143, 'surprisePercent': 4.1934, 'symbol': 'AAPL', 'year': 2026}
{'actual': 1.85, 'estimate': 1.8075, 'period': '2025-09-30', 'quarter': 4, 'surprise': 0.0425, 'surprisePercent': 2.3513, 'symbol': 'AAPL', 'year': 2025}
{'actual': 1.57, 'estimate': 1.4626, 'period': '2025-06-30', 'quarter': 3, 'surprise': 0.1074, 'surprisePercent': 7.3431, 'symbol': 'AAPL', 'year': 2025}
{'actual': 1.65, 'estimate': 1.6596, 'period': '2025-03-31', 'quarter': 2, 'surprise': -0.0096, 'surprisePercent': -0.5785, 'symbol': 'AAPL', 'year': 2025}


### ─────────────────────────────────────────────
### SECTION 2 — PRODUCTION RUN (ALL 60 COMPANIES)
### ─────────────────────────────────────────────

In [8]:
import finnhub
import os
import time
from dotenv import load_dotenv

load_dotenv()
finnhub_client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

tickers = [
    # DEFENSE - High Lobby
    "LMT", "RTX", "NOC", "GD", "BA", "LHX", "LDOS", "HII", "BAESY", "SAIC",
    # DEFENSE - Low Lobby
    "TXT", "TDG", "HEI", "DRS", "KTOS", "AVAV", "MRCY", "CW", "MOG.A", "DCO",
    # ENERGY - High Lobby
    "XOM", "CVX", "COP", "OXY", "BP", "NEE", "D", "DUK", "HAL", "BKR",
    # ENERGY - Low Lobby
    "SLB", "VLO", "PSX", "EOG", "FANG", "DVN", "CTRA", "AR", "CHRD", "MTDR",
    # TECH - High Lobby
    "MSFT", "AMZN", "GOOGL", "IBM", "ORCL", "PLTR", "BAH", "CACI", "PSN", "CRM",
    # TECH - Low Lobby
    "AAPL", "META", "NVDA", "CSCO", "PANW", "CRWD", "SNOW", "DDOG", "NET", "TWLO"
]

# Duplicate guard
duplicates = [t for t in tickers if tickers.count(t) > 1]
assert not duplicates, f"Duplicate tickers found: {duplicates}"

results = {}
issues = []

for i, ticker in enumerate(tickers):  # fixed: ticker not tickers
    try:
        data = finnhub_client.company_earnings(symbol=ticker)
        results[ticker] = data

        # Check for empty response
        if not data:
            issues.append((ticker, "empty response — ticker may be unavailable"))
            continue

        # Check number of quarters returned
        if len(data) < 4:
            issues.append((ticker, f"fewer than 4 quarters returned: {len(data)}"))

        # Check for None fields within each quarter
        for quarter in data:
            none_fields = [k for k, v in quarter.items() if v is None or v == ""]
            if none_fields:
                issues.append((ticker, f"missing fields in {quarter.get('period')}: {none_fields}"))

    except Exception as e:
        issues.append((ticker, f"API error: {str(e)}"))
        results[ticker] = []

    if i < len(tickers) - 1:
        time.sleep(1)

# Summary
successful = [t for t, r in results.items() if r]
print(f"✅ Successfully pulled: {len(successful)} / {len(tickers)} companies")
print(f"⚠️  Issues found: {len(issues)}")
for ticker, issue in issues:
    print(f"   {ticker}: {issue}")

empty = [ticker for ticker, data in results.items() if not data]
print(f"\nEmpty responses: {empty if empty else 'None'}")

✅ Successfully pulled: 60 / 60 companies
⚠️  Issues found: 0

Empty responses: None
